In [30]:
import csv

import numpy as np
import pandas as pd
from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord
import astropy.units as u
from astroplan import Observer, FixedTarget
from astroquery.simbad import Simbad
from geopy.geocoders import Nominatim

import warnings
from astroplan import TargetAlwaysUpWarning
# Suppress just this specific warning
warnings.filterwarnings('ignore', category=TargetAlwaysUpWarning)



In [31]:
def azimuth_to_cardinal(azimuth, points=8):
    """
    Converts azimuth degrees (0-360) into standard cardinal points.
    Accepts points=4 (N, E, S, W) or points=8 (N, NE, E, SE, S, SW, W, NW).
    """
    # Ensure azimuth stays within 0 to 360 degrees
    azimuth = azimuth % 360
    
    if points == 4:
        directions = ["N", "E", "S", "W"]
    elif points == 8:
        directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    else:
        raise ValueError("Points parameter must be 4 or 8.")
        
    # Calculate index by dividing 360 degrees into equal wedges
    num_directions = len(directions)
    wedge_size = 360 / num_directions
    
    # Add half a wedge size to center the degree range on the cardinal point
    idx = int((azimuth + (wedge_size / 2)) / wedge_size) % num_directions
    
    return directions[idx]

# --- Quick Test ---
#test_degrees = [0, 45, 90, 185, 270, 355]

#print("8-Point Conversion:")
#for deg in test_degrees:
#    print(f"{deg}° -> {azimuth_to_cardinal(deg, points=8)}")

In [32]:


def is_target_always_up(observer, target, time, horizon_deg=0.0):
    """
    Checks if a target stays above the horizon for 24 hours starting from 'time'.
    """
    # 1. Create a grid of times covering 24 hours
    time_grid = time + np.linspace(0, 24, 96) * u.hour
    
    # 2. Get the AltAz coordinates (returns a SkyCoord object)
    altaz_coords = observer.altaz(time_grid, target)
    
    # 3. Extract just the altitudes
    altitudes = altaz_coords.alt
    
    # 4. Find the minimum altitude reached during the day
    min_altitude = np.min(altitudes)
    
    # 5. If the lowest point is above the horizon, it's always up
    return min_altitude > horizon_deg * u.deg
    

In [33]:
# Initialize the geocoder (provide a unique app name for the user_agent)
geolocator = Nominatim(user_agent="my_location_app")

# Define the address you want to find
address = "Wiesenstrasse 58 64331 Weiterstadt Germany"
location = geolocator.geocode(address)

# 2. Define Observation Time
# Input the specific year, month, day, and UTC time
obs_time = Time("2026-06-27 22:00:00") # Format: YYYY-MM-DD HH:MM:SS (UTC)
formatted_time = obs_time.strftime("%Y_%m_%d_%H%M")


In [34]:

# Check if a location was found and print the coordinates
if location:
    print(f"Address: {location.address}")
    print(f"Latitude: {location.latitude}, Longitude: {location.longitude}")
    print(location)
else:
    print("Address not found.")

# 1. Define Observation Location (Example: Reichelsheim, Germany)
# Replace with your exact latitude, longitude, and elevation
location = EarthLocation(lat=location.latitude*u.deg, lon=location.longitude*u.deg, height=110*u.m)
observer = Observer(location=location, name="Home Observatory")


with open("targets.txt", newline="", encoding="utf-8") as csvfile:
    target_list = list(csv.DictReader(csvfile))


# 4. Calculate Coordinates and Build Database
db_records = []

for item in target_list:
    try:
        # Fetch target coordinates from online catalogs (SIMBAD/NED)
        target = FixedTarget.from_name(item["id"], name=item["name"])
        # Calculate horizontal coordinates (Alt/Az) for the observer at that time
        altaz = observer.altaz(obs_time, target)
        
        # Extract components
        altitude = altaz.alt.degree
        azimuth = altaz.az.degree
        zenith = 90.0 - altitude # Zenith angle is the complement of altitude
        
        # Get Rise and Fall (Set) Times
        rise_time = observer.target_rise_time(obs_time, target, which="next")
        set_time = observer.target_set_time(obs_time, target, which="next")
        always_up = is_target_always_up(observer, target, obs_time, horizon_deg=0.0)
       
        Simbad.add_votable_fields('dim')  
        result_table = Simbad.query_object(item["id"])
        
        # Extract major and minor dimensions
        maj_axis = result_table['GALDIM_MAJAXIS'][0]
        min_axis = result_table['GALDIM_MINAXIS'][0]
        unit = result_table['GALDIM_MAJAXIS'].unit
                
        # Append data row
        db_records.append({
            "Target Name": target.name,
            "RA (deg)": target.coord.ra.degree,
            "Dec (deg)": target.coord.dec.degree,
            "Azimuth (deg)": round(azimuth, 2),
            "Cardinal ": azimuth_to_cardinal(azimuth, points=8),
            "Altitude (deg)": round(altitude, 2),
            "Zenith (deg)": round(zenith, 2),
            "major_axis ": maj_axis,
            "major_axis_units ": result_table['GALDIM_MAJAXIS'].unit,
            "minor_axis": min_axis,
            "minor_axis_units ": result_table['GALDIM_MINAXIS'].unit,
            "Rise Time": rise_time.iso,
            "Set  Time": set_time.iso,           
            "Circumpolar": always_up,
            "Visible Now": altitude > 0 # True if above the horizon
        })
    except Exception as e:
        print(f"Could not fetch data for {item['name']}: {e}")

# 5. Convert to Pandas DataFrame
df = pd.DataFrame(db_records)

# Display the resulting database table
print(f"\n--- DSO Database for {obs_time} UTC ---")
print(df.to_string(index=False))

# 6. Optional: Save database to a CSV file
print(f"...saving weiterstadt_dso_database_{formatted_time}.csv")
df.to_csv(f"weiterstadt_dso_database_{formatted_time}.csv", index=False)
print('...processing finished')

Address: 58, Wiesenstraße, Riedbahn, Weiterstadt, Landkreis Darmstadt-Dieburg, Hessen, 64331, Deutschland
Latitude: 49.8866755, Longitude: 8.6095692
58, Wiesenstraße, Riedbahn, Weiterstadt, Landkreis Darmstadt-Dieburg, Hessen, 64331, Deutschland
Could not fetch data for NGC7635 (Bubble Nenula) NGC7635: quote_from_bytes() expected bytes

--- DSO Database for 2026-06-27 22:00:00.000 UTC ---
               Target Name   RA (deg)  Dec (deg)  Azimuth (deg) Cardinal   Altitude (deg)  Zenith (deg) major_axis  major_axis_units  minor_axis minor_axis_units                Rise Time               Set  Time  Circumpolar  Visible Now
NGC6960/C34 (Western Veil) 311.408333  30.708333          92.17         E           43.84         46.16          --            arcmin         --            arcmin 2026-06-28 16:44:53.576 2026-06-28 10:46:20.541        False         True
NGC6992/C33 (Eastern Veil) 314.079167  31.743333          89.03         E           42.80         47.20          --            arcmin 